# NB2 · Fine-tuning: completo vs. eficiente (LoRA / QLoRA)

**Inteligencia Artificial y Aprendizaje Profundo II — Unidad IV**  
Prof. José Canelón, Ph.D.

Notebook **pesado**: requiere **GPU**. La Parte A (encoder chico) corre en Colab **Free (T4)**; la Parte B (QLoRA) corre en Free con un modelo chico, y admite Mistral-7B en **Colab Pro**.

**Contenido:** Parte A — full fine-tuning **vs.** LoRA (comparando parámetros, memoria y tamaño del checkpoint). Parte B — **QLoRA** en 4 bits sobre un LLM, con `SFTTrainer`, y merge/guardar/publicar.

Las librerías del ecosistema (trl, peft, bitsandbytes) evolucionan rápido. Si alguna celda falla por un cambio de API, suele bastar un ajuste menor según la versión instalada.

## 0. Preparación (GPU obligatoria)

In [1]:
import torch
assert torch.cuda.is_available(), 'Activa la GPU: Entorno de ejecución → Cambiar tipo de entorno → T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

GPU: Tesla T4


In [2]:
%pip install -q -U transformers datasets accelerate peft bitsandbytes trl evaluate
# Colab suele traer una versión vieja de torchao que rompe a peft; no la usamos (cuantizamos con
# bitsandbytes), así que la quitamos. Al estar al inicio, evita el error sin necesidad de reiniciar.
%pip uninstall -q -y torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 110.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.8 MB/s eta 0:00:00


In [3]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/Colab_Notebooks/UCA/IA_DL_II/Unidad_IV'
except Exception:
    BASE = './UCA/IA_DL_II/Unidad_IV'
import os; os.makedirs(BASE, exist_ok=True); print('BASE =', BASE)

Mounted at /content/drive
BASE = /content/drive/MyDrive/Colab_Notebooks/UCA/IA_DL_II/Unidad_IV


# PARTE A — Full fine-tuning vs. LoRA

Misma tarea (clasificación de reseñas) y mismo modelo base (`distilbert-base-uncased`), dos formas de adaptarlo. Vamos a **medir** la diferencia en parámetros entrenables, memoria y tamaño del checkpoint.

In [4]:
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
import numpy as np, evaluate, os

ckpt = 'distilbert-base-uncased'
ds = load_dataset('cornell-movie-review-data/rotten_tomatoes')
tok = AutoTokenizer.from_pretrained(ckpt)
ds_tok = ds.map(lambda b: tok(b['text'], truncation=True), batched=True)
collator = DataCollatorWithPadding(tok)
acc = evaluate.load('accuracy')
def compute_metrics(p):
    return acc.compute(predictions=np.argmax(p.predictions, axis=1), references=p.label_ids)
def n_trainable(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
def dir_size_mb(path):
    tot = sum(os.path.getsize(os.path.join(dp,f)) for dp,_,fs in os.walk(path) for f in fs)
    return round(tot/1e6, 1)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/7.46k [00:00<?, ?B/s]

train.parquet: reconstructing file:   0%|          |  0.00B /  699kB            

train.parquet: downloading bytes:           |  0.00B            

validation.parquet: reconstructing file:   0%|          |  0.00B / 90.0kB            

validation.parquet: downloading bytes:           |  0.00B            

... (8 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

### A.1 — Full fine-tuning (se actualizan **todos** los pesos)

In [5]:
model_full = AutoModelForSequenceClassification.from_pretrained(ckpt, num_labels=2)
print('Parámetros entrenables (full):', f'{n_trainable(model_full):,}')

args_full = TrainingArguments(f'{BASE}/out_full', per_device_train_batch_size=16,
    num_train_epochs=1, learning_rate=2e-5, eval_strategy='epoch', report_to='none', fp16=True)
tr_full = Trainer(model_full, args_full, train_dataset=ds_tok['train'], eval_dataset=ds_tok['test'],
                  data_collator=collator, compute_metrics=compute_metrics)
tr_full.train()
eval_full = tr_full.evaluate(); print(eval_full)
tr_full.save_model(f'{BASE}/ckpt_full')

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parámetros entrenables (full): 66,955,010


Epoch,Training Loss,Validation Loss


... (2 outputs left out)

Epoch,Training Loss,Validation Loss,Accuracy
1,0.416002,0.384938,0.837711


Training Loss,Validation Loss,Epoch,Accuracy
0.416002,0.384938,1,0.837711


{'eval_loss': 0.38493818044662476, 'eval_accuracy': 0.8377110694183865}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

### A.2 — LoRA (se congela el modelo; se entrenan matrices de bajo rango)

`r` = rango, `lora_alpha` = escala, `target_modules` = dónde inyectar (en DistilBERT: `q_lin`, `v_lin`).

In [6]:
from peft import LoraConfig, get_peft_model, TaskType

model_lora = AutoModelForSequenceClassification.from_pretrained(ckpt, num_labels=2)
lora_cfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.05,
                      target_modules=['q_lin', 'v_lin'])
model_lora = get_peft_model(model_lora, lora_cfg)
model_lora.print_trainable_parameters()   # ¡mirá el porcentaje!

args_lora = TrainingArguments(f'{BASE}/out_lora', per_device_train_batch_size=16,
    num_train_epochs=1, learning_rate=1e-3, eval_strategy='epoch', report_to='none', fp16=True)
tr_lora = Trainer(model_lora, args_lora, train_dataset=ds_tok['train'], eval_dataset=ds_tok['test'],
                  data_collator=collator, compute_metrics=compute_metrics)
tr_lora.train()
eval_lora = tr_lora.evaluate(); print(eval_lora)
model_lora.save_pretrained(f'{BASE}/ckpt_lora')   # guarda SOLO el adaptador

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy
1,0.414286,0.389967,0.835835


Training Loss,Validation Loss,Epoch,Accuracy
0.414286,0.389967,1,0.835835


{'eval_loss': 0.38996726274490356, 'eval_accuracy': 0.8358348968105066}


### A.3 — Comparación

Lo importante: LoRA logra calidad comparable entrenando **una fracción ínfima** de parámetros, y su checkpoint pesa **muchísimo menos** (solo el adaptador).

In [7]:
import pandas as pd
tabla = pd.DataFrame([
  {'método':'full', 'accuracy': round(eval_full['eval_accuracy'],4),
   'params_entrenables': n_trainable(model_full), 'checkpoint_MB': dir_size_mb(f'{BASE}/ckpt_full')},
  {'método':'LoRA', 'accuracy': round(eval_lora['eval_accuracy'],4),
   'params_entrenables': n_trainable(model_lora), 'checkpoint_MB': dir_size_mb(f'{BASE}/ckpt_lora')},
])
tabla

,método,accuracy,params_entrenables,checkpoint_MB
0,full,0.8377,66955010,268.6
1,LoRA,0.8358,739586,3.0


# PARTE B — QLoRA sobre un LLM

Cuantizamos el modelo base a **4 bits (NF4)** y entrenamos adaptadores **LoRA** encima. Solo el adaptador recibe gradientes.

In [8]:
# Modelo base. Por defecto uno CHICO y sin gating (corre en Colab Free).
MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
# Para Mistral-7B (requiere Colab Pro + aceptar la licencia en el Hub + token):
# MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.3'
# Alternativa intermedia: 'Qwen/Qwen2.5-3B-Instruct'

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
tokL = AutoTokenizer.from_pretrained(MODEL_ID)
if tokL.pad_token is None: tokL.pad_token = tokL.eos_token
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map='auto')
print('Modelo base cargado en 4 bits.')

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Modelo base cargado en 4 bits.


In [11]:
# Prompt de prueba ANTES del fine-tuning
def responder(model, instruccion, max_new_tokens=80):
    msgs = [{'role': 'user', 'content': instruccion}]
    inputs = tokL.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors='pt', return_dict=True
    ).to(model.device)
    input_len = inputs['input_ids'].shape[1]
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokL.decode(out[0][input_len:], skip_special_tokens=True)

print(responder(base, 'Explicá en una frase qué es el fine-tuning eficiente en parámetros.'))

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


El finetuning eficiente en parámetros es una técnica de aprendizaje automático que se basa en la selección de parámetros de una función de aprendizaje automático que mejoran su eficiencia en el proceso de aprendizaje. En términos generales, el finetuning es una técnica de aprendiz


In [12]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

model = prepare_model_for_kbit_training(base)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                  target_modules=['q_proj','v_proj'], task_type='CAUSAL_LM')
model = get_peft_model(model, lora)
model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [13]:
# Dataset de instrucciones (subconjunto chico para la demo)
from datasets import load_dataset
raw = load_dataset('yahma/alpaca-cleaned', split='train[:512]')

def to_text(ex):
    user = ex['instruction'] + (('\n' + ex['input']) if ex['input'] else '')
    msgs = [{'role':'user','content': user}, {'role':'assistant','content': ex['output']}]
    return {'text': tokL.apply_chat_template(msgs, tokenize=False)}

ds_sft = raw.map(to_text, remove_columns=raw.column_names)
print(ds_sft[0]['text'][:300])

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

alpaca_data_cleaned.json: reconstructing file:   0%|          |  0.00B / 44.3MB            

alpaca_data_cleaned.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Map:   0%|          | 0/512 [00:00<?, ? examples/s]

<|user|>
Give three tips for staying healthy.</s>
<|assistant|>
1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at i


In [15]:
from trl import SFTTrainer, SFTConfig

cfg = SFTConfig(output_dir=f'{BASE}/out_qlora',
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    num_train_epochs=1, learning_rate=2e-4, bf16=True,
    logging_steps=10, max_length=512, dataset_text_field='text',
    gradient_checkpointing=True, report_to='none')
trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds_sft)
trainer.train()

Adding EOS to train dataset:   0%|          | 0/512 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/512 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/512 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/512 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/512 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,1.492250
20,1.306853
30,1.244163
40,1.270576
50,1.176564
60,1.108658


TrainOutput(global_step=64, training_loss=1.2688159942626953, metrics={'train_runtime': 414.1461, 'train_samples_per_second': 1.236, 'train_steps_per_second': 0.155, 'total_flos': 819960995782656.0, 'train_loss': 1.2688159942626953, 'entropy': 1.3451749831438065, 'num_tokens': 92895.0, 'mean_token_accuracy': 0.6946316137909889, 'epoch': 1.0})

In [16]:
# Prompt de prueba DESPUÉS del fine-tuning
print(responder(model, 'Explicá en una frase qué es el fine-tuning eficiente en parámetros.'))

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.


Elms.
<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<


In [17]:
# Guardar el adaptador (pocos MB) y, opcionalmente, publicarlo
model.save_pretrained(f'{BASE}/adaptador_qlora')
print('Adaptador guardado:', dir_size_mb(f'{BASE}/adaptador_qlora'), 'MB')
# from huggingface_hub import login; login(token='hf_...')
# model.push_to_hub('tu-usuario/mi-adaptador-lora')

Adaptador guardado: 4.5 MB


### Nota sobre `merge`

`merge_and_unload()` funde LoRA en los pesos base y da un modelo autónomo **sin latencia extra**, pero **no** aplica directamente sobre un base en 4 bits: primero hay que recargar el modelo en 16 bits (sin cuantizar), aplicarle el adaptador y ahí fusionar. Para servir, muchas veces conviene dejar el adaptador separado (un backbone, muchos adaptadores).

In [ ]:
# (Opcional) merge en 16 bits
# from peft import PeftModel
# base16 = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto')
# merged = PeftModel.from_pretrained(base16, f'{BASE}/adaptador_qlora').merge_and_unload()
# merged.save_pretrained(f'{BASE}/modelo_fusionado')